# Final iTransformer with MOE and Quntum Computing

In [2]:
import os
import gc
import warnings
import logging
from itertools import cycle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau, LambdaLR
from tqdm import tqdm
from scipy.signal import butter, filtfilt, find_peaks
from scipy.interpolate import interp1d
from scipy.stats import pearsonr
import pywt
import pennylane as qml
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_fscore_support
from sklearn.preprocessing import PowerTransformer, label_binarize

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Quiet some transformer warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.nn.modules.transformer")

# CUDA / PyTorch performance knobs
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
torch.set_float32_matmul_precision("high")

# Path to dataset CSV
DATASET_PATH = "/kaggle/input/two-lead-test/DATASET_CSV.csv"

# ---------------------------
# Signal processing utilities
# ---------------------------
def bandpass_filter(data, lowcut=0.3, highcut=50.0, fs=360.0, order=5):
    """Apply bandpass filter to ECG signal"""
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype="band")
    filtered = filtfilt(b, a, data, axis=0)
    return filtered

def mad(data):
    """Median Absolute Deviation"""
    return np.median(np.abs(data - np.median(data)))

def wavelet_denoise(data, wavelet="db8", level=4):
    """Wavelet denoising for ECG signals"""
    coeffs = pywt.wavedec(data, wavelet, level=level)
    sigma = mad(coeffs[-level])
    uthresh = sigma * np.sqrt(2 * np.log(len(data)))
    for i in range(1, len(coeffs)):
        coeffs[i] = pywt.threshold(coeffs[i], value=uthresh, mode="soft")
    return pywt.waverec(coeffs, wavelet)

def align_r_peaks(data, fs=360.0, window=250):
    """Align ECG signals based on R-peaks"""
    aligned_data = []
    for i in range(0, len(data) - window, window):
        segment = data[i : i + window]
        peaks, _ = find_peaks(segment[:, 0], height=0.5, distance=int(fs / 3))
        if len(peaks) > 0:
            r_peak = peaks[0]  # Take first peak for simplicity
            start = max(0, r_peak - window // 2)
            end = start + window
            if end <= len(segment):
                aligned_data.append(segment[start:end])
            else:
                aligned_data.append(segment[-window:])
        else:
            aligned_data.append(segment)
    return np.array(aligned_data).reshape(-1, data.shape[1])

def add_baseline_wander(data, amplitude=0.1, frequency=0.5, fs=360.0):
    """Add baseline wander artifact"""
    t = np.arange(len(data)) / fs
    wander = amplitude * np.sin(2 * np.pi * frequency * t)
    return data + wander[:, np.newaxis]

def add_muscle_artifact(data, amplitude=0.05, fs=360.0):
    """Add muscle artifact noise"""
    t = np.arange(len(data)) / fs
    noise = amplitude * np.random.randn(len(data))
    return data + noise[:, np.newaxis]

# ---------------------------
# Model components
# ---------------------------
class MoE(nn.Module):
    """Mixture of Experts layer"""
    def __init__(self, d_model, num_experts, d_ffn, dropout=0.2):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ffn),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_ffn, d_model),
                nn.Dropout(dropout)
            )
            for _ in range(num_experts)
        ])
        self.router = nn.Linear(d_model, num_experts)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        router_logits = self.router(x)
        router_weights = F.softmax(router_logits, dim=-1)
        expert_outputs = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            expert_mask = router_weights[:, :, i:i+1]
            expert_output = expert(x)
            expert_outputs += expert_mask * expert_output
        return self.dropout(expert_outputs)

class InvertedTransformerEncoderLayer(nn.Module):
    """Inverted Transformer Encoder Layer with MoE"""
    def __init__(self, d_model, num_heads, dim_feedforward, num_experts=8, dropout=0.1, activation="gelu"):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.moe = MoE(d_model, num_experts, dim_feedforward, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.activation = F.gelu if activation == "gelu" else F.relu

    def forward(self, src):
        # src: [batch, num_variates, seq_len, d_model]
        batch_size, num_variates, seq_len, d_model = src.shape
        attn_input = src.permute(0, 2, 1, 3).reshape(batch_size * seq_len, num_variates, d_model)
        attn_output, _ = self.self_attn(attn_input, attn_input, attn_input)
        attn_output = self.dropout1(attn_output)
        attn_output = self.norm1(attn_input + attn_output)
        attn_output = attn_output.reshape(batch_size, seq_len, num_variates, d_model).permute(0, 2, 1, 3)
        moe_input = attn_output.reshape(batch_size, num_variates, -1)
        moe_output = self.moe(moe_input)
        moe_output = moe_output.reshape(batch_size, num_variates, seq_len, d_model)
        return self.norm2(attn_output + moe_output)

class QuantumCircuitWrapper:
    """Wrapper for quantum circuit with robust error handling"""
    def __init__(self, num_qubits, num_layers, device="cuda:0"):
        self.num_qubits = num_qubits
        self.num_layers = num_layers
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        
        # Initialize quantum device
        try:
            self.dev = qml.device("lightning.gpu", wires=self.num_qubits)
            logger.info("Using lightning.gpu device for quantum circuit")
        except Exception as e:
            logger.warning(f"Failed to initialize lightning.gpu: {e}")
            self.dev = qml.device("default.qubit", wires=self.num_qubits)
            logger.info("Falling back to default.qubit device for quantum circuit")
        
        # Fallback classical layer
        self.fallback_layer = nn.Linear(self.num_qubits, self.num_qubits).to(self.device)
        
        # Quantum weights
        weight_shapes = (self.num_layers, self.num_qubits, 2)
        self.weights = nn.Parameter(torch.randn(weight_shapes) * 0.1, requires_grad=True).to(self.device)
        
        # Define quantum circuit
        self.quantum_circuit = self._create_quantum_circuit()

    def _create_quantum_circuit(self):
        """Create quantum circuit with robust error handling"""
        @qml.qnode(self.dev, interface="torch", diff_method="parameter-shift")
        def quantum_circuit(inputs, weights):
            # Robust input normalization
            inputs = F.normalize(inputs, p=2, dim=-1)
            inputs = torch.clamp(inputs, -1.0, 1.0)
            
            if torch.any(torch.isnan(inputs)) or torch.any(torch.isinf(inputs)):
                inputs = torch.zeros_like(inputs)
                logger.warning("NaN/Inf detected in quantum circuit inputs - zeroing")
            
            try:
                for layer in range(self.num_layers):
                    for i in range(self.num_qubits):
                        qml.RX(inputs[:, i], wires=i)
                        qml.RY(weights[layer, i, 0], wires=i)
                        qml.RZ(weights[layer, i, 1], wires=i)
                    for i in range(self.num_qubits - 1):
                        qml.CNOT(wires=[i, i + 1])
                measurements = [qml.expval(qml.PauliZ(i)) for i in range(self.num_qubits)]
                return torch.stack(measurements, dim=1)
            except Exception as e:
                logger.error(f"Quantum circuit execution failed: {e}")
                # Return fallback values
                return torch.zeros(inputs.shape[0], self.num_qubits, device=inputs.device)
        
        return quantum_circuit

    def forward(self, inputs):
        """Execute quantum circuit with fallback"""
        try:
            # Ensure inputs are on correct device and dtype
            inputs = inputs.to(self.device, dtype=torch.float32)
            weights = self.weights.to(self.device, dtype=torch.float32)
            
            outputs = self.quantum_circuit(inputs, weights)
            
            if torch.any(torch.isnan(outputs)) or torch.any(torch.isinf(outputs)):
                logger.warning("NaN/Inf in quantum outputs - using fallback")
                outputs = self.fallback_layer(inputs)
                
            return outputs
            
        except Exception as e:
            logger.error(f"Quantum circuit failed completely: {e}")
            return self.fallback_layer(inputs)

class iTransformer(nn.Module):
    """Inverted Transformer with Quantum Enhancement"""
    def __init__(
        self,
        num_variates,
        seq_len,
        pred_len,
        n_classes,
        hidden_size=512,
        num_heads=8,
        num_layers=4,
        num_experts=8,
        dropout=0.3,
        ffn_multiplier=4,
        activation="gelu",
        device="cuda:0",
    ):
        super().__init__()
        self.num_variates = num_variates
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.hidden_size = hidden_size
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        
        # Input projection
        self.input_projection = nn.Linear(seq_len, hidden_size)
        self.variate_pos_embed = nn.Parameter(torch.randn(1, num_variates, 1, hidden_size))
        
        # Transformer layers
        self.layers = nn.ModuleList([
            InvertedTransformerEncoderLayer(
                d_model=hidden_size,
                num_heads=num_heads,
                dim_feedforward=ffn_multiplier * hidden_size,
                num_experts=num_experts,
                dropout=dropout,
                activation=activation
            )
            for _ in range(num_layers)
        ])
        
        # Heads
        self.quantum_head = nn.Linear(hidden_size * num_variates, hidden_size)
        self.classifier_head = nn.Linear(hidden_size * num_variates, hidden_size)
        self.quantum_output = nn.Linear(hidden_size, 1)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_size // 2, n_classes)
        )
        
        # Quantum branch
        self.num_qubits = 16
        self.quantum_wrapper = QuantumCircuitWrapper(self.num_qubits, num_layers=5, device=device)
        self.quantum_proj = nn.Linear(num_variates, self.num_qubits)
        self.quantum_dropout = nn.Dropout(dropout)
        self.quantum_post = nn.Linear(self.num_qubits, 1)
        self.quantum_scale = nn.Parameter(torch.tensor(5.0))

    def validate_shapes(self, x):
        """Validate input shapes throughout the model"""
        expected_shape = (x.shape[0], self.seq_len, self.num_variates)
        if x.shape != expected_shape:
            raise ValueError(f"Expected input shape {expected_shape}, got {x.shape}")
        return True

    def forward(self, x):
        # Validate input shape
        self.validate_shapes(x)
        
        batch_size, seq_len, num_variates = x.shape
        
        # [batch, variate, seq]
        x = x.permute(0, 2, 1)
        
        with torch.amp.autocast("cuda", dtype=torch.float16):
            x_proj = self.input_projection(x)         # -> [batch, variate, hidden]
            x_proj = x_proj.unsqueeze(2)              # -> [batch, variate, 1, hidden]
            x_proj = x_proj + self.variate_pos_embed  # add positional
            
            for layer in self.layers:
                x_proj = layer(x_proj)
                
            x_flat = x_proj.reshape(batch_size, -1)
        
        # Quantum branch
        quantum_branch = self.quantum_head(x_flat)
        
        with torch.amp.autocast("cuda", dtype=torch.float16):
            quantum_out = self.quantum_output(quantum_branch)
            
            # Prepare inputs for quantum circuit
            inputs_quantum = x[:, :, -1]  # [batch, variate]
            inputs_quantum = self.quantum_proj(inputs_quantum)
            
            # Execute quantum circuit
            quantum_outputs = self.quantum_wrapper.forward(inputs_quantum)
            quantum_outputs = self.quantum_dropout(quantum_outputs)
            
            # Combine quantum and classical outputs
            pred_quantum = quantum_out + self.quantum_post(quantum_outputs) * self.quantum_scale
        
        # Classification branch
        class_branch = self.classifier_head(x_flat.float())
        class_logits = self.classifier(class_branch)
        
        return pred_quantum.flatten(), class_logits

# ---------------------------
# Loss and dataset
# ---------------------------
class ECG_loss(nn.Module):
    """Combined loss function for quantum regression and classification"""
    def __init__(self, quantum_weight=0.4, class_weight=0.6):
        super().__init__()
        self.huber = nn.HuberLoss()
        self.class_weights = torch.tensor([1.0, 1.0, 1.0, 1.0], dtype=torch.float32)
        self.cross_entropy = nn.CrossEntropyLoss(weight=self.class_weights)
        self.quantum_weight = quantum_weight
        self.class_weight = class_weight

    def forward(self, pred_quantum, target_quantum, pred_logits, target_class):
        quantum_loss = self.huber(pred_quantum, target_quantum)
        class_loss = self.cross_entropy(pred_logits, target_class)
        return self.quantum_weight * quantum_loss + self.class_weight * class_loss

class TimeSeriesDataset(Dataset):
    """Dataset for ECG time series with augmentation"""
    def __init__(self, data, seq_len, pred_len, labels=None):
        self.features = data.values.astype(np.float32)
        self.labels = labels.astype(np.int64) if labels is not None else None
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_samples = len(data) - seq_len - pred_len + 1
        
        if self.num_samples <= 0:
            raise ValueError(f"Insufficient data: num_samples = {self.num_samples}")
        
        # Precompute augmented data
        self.augmented = self._precompute_augmentations()
        
        # Quantum targets
        all_data = self.features[self.seq_len : self.seq_len + self.num_samples, :]
        self.quantum_targets = np.mean(all_data, axis=1).astype(np.float32)
        
        if self.labels is not None:
            self.labels = self.labels[self.seq_len : self.seq_len + self.num_samples]

    def _precompute_augmentations(self):
        """Precompute all augmentations for better performance"""
        augmented = []
        rng = np.random.default_rng()
        
        for idx in range(self.num_samples):
            x = self.features[idx : idx + self.seq_len]
            
            # Apply augmentations
            x = add_baseline_wander(x)
            x = add_muscle_artifact(x)
            
            # Time warping
            warp_factor = rng.uniform(0.9, 1.1)
            time_orig = np.linspace(0, 1, self.seq_len)
            time_warped = np.linspace(0, 1, self.seq_len) * warp_factor
            time_warped = time_warped / time_warped[-1] * time_orig[-1]
            
            x_warped = np.zeros_like(x)
            for v in range(x.shape[1]):
                interp = interp1d(time_orig, x[:, v], kind="linear", fill_value="extrapolate")
                x_warped[:, v] = interp(time_warped)
            
            # Additional noise
            noise1 = rng.normal(0, 0.1, x_warped.shape).astype(np.float32)
            scale = rng.uniform(0.8, 1.2)
            noise2 = rng.normal(0, 0.05, x_warped.shape).astype(np.float32)
            
            x = x_warped * scale + noise1 + noise2
            
            # Random cropping
            if rng.random() > 0.5:
                crop_start = rng.integers(0, self.seq_len // 10)
                if crop_start + self.seq_len <= len(x):
                    x = x[crop_start : crop_start + self.seq_len]
            
            # Clamp and clean
            x = np.clip(x, -5, 5)
            x = np.nan_to_num(x, nan=0.0, posinf=5.0, neginf=-5.0)
            
            # Ensure correct shape
            if x.shape[0] != self.seq_len:
                x = x[:self.seq_len]  # truncate if needed
                if x.shape[0] < self.seq_len:
                    # Pad if necessary
                    pad_width = self.seq_len - x.shape[0]
                    x = np.pad(x, ((0, pad_width), (0, 0)), mode='edge')
            
            augmented.append(x)
        
        return augmented

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        idx = idx % self.num_samples
        x = self.augmented[idx]
        y_quantum = self.quantum_targets[idx]
        
        # Handle labels properly
        if self.labels is not None:
            y_label = self.labels[idx]
        else:
            # For inference, use dummy label
            y_label = 0
        
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y_quantum, dtype=torch.float32),
            torch.tensor(y_label, dtype=torch.int64),
        )

# ---------------------------
# Metrics and helpers
# ---------------------------
def calculate_metrics(true_quantum, pred_quantum):
    """Calculate regression metrics for quantum predictions"""
    rmse = torch.sqrt(torch.mean((true_quantum - pred_quantum) ** 2))
    mae = torch.mean(torch.abs(true_quantum - pred_quantum))
    std_error = torch.std(true_quantum - pred_quantum)
    z_score = 1.96
    conf_interval = z_score * std_error / np.sqrt(true_quantum.size(0))
    
    true_range = torch.clamp(torch.max(true_quantum) - torch.min(true_quantum), min=1e-6)
    threshold = 0.3 * true_range
    abs_error = torch.abs(true_quantum - pred_quantum)
    accuracy_percentage = torch.mean((abs_error < threshold).float()) * 100
    
    # Pearson correlation coefficient
    corr_coef, _ = pearsonr(true_quantum.cpu().numpy(), pred_quantum.cpu().numpy())
    
    return {
        "Quantum_RMSE": rmse.item(),
        "Quantum_MAE": mae.item(),
        "Quantum_Conf_Interval": conf_interval.item(),
        "Quantum_Accuracy_Percentage": accuracy_percentage.item(),
        "Quantum_Correlation": corr_coef,
    }

def get_optimizer(model, optimizer_type, lr, weight_decay):
    """Get optimizer with specified configuration"""
    if optimizer_type.lower() == "adamw":
        return optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
            betas=(0.9, 0.999),
            eps=1e-8,
        )
    elif optimizer_type.lower() == "adam":
        return optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
            betas=(0.9, 0.999),
            eps=1e-8,
        )
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_type}")

def get_scheduler(optimizer, scheduler_type, warmup_epochs, total_epochs):
    """Get learning rate scheduler"""
    if scheduler_type.lower() == "cosine":
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return (epoch + 1) / warmup_epochs
            return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (total_epochs - warmup_epochs)))
        return LambdaLR(optimizer, lr_lambda)
    elif scheduler_type.lower() == "reduce_on_plateau":
        return ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=5, verbose=True)
    else:
        raise ValueError(f"Unsupported scheduler: {scheduler_type}")

def clear_memory():
    """Clear GPU memory and garbage collect"""
    torch.cuda.empty_cache()
    gc.collect()

# ---------------------------
# Main training routine
# ---------------------------
def main():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device: {device}")
    # Load
    try:
        raw_data = pd.read_csv(DATASET_PATH)
        raw_data.replace([np.inf, -np.inf], np.nan, inplace=True)
        raw_data.dropna(inplace=True)
        print(f"Raw data shape after dropping NaN: {raw_data.shape}")
    except Exception as e:
        raise ValueError(f"Error loading dataset: {e}")
    # Expect 'type' labels
    if "type" not in raw_data.columns:
        raise ValueError("Dataset must contain 'type' column for classification labels")
    feature_columns = [col for col in raw_data.columns if col != "type"]
    labels = raw_data["type"].values
    features = raw_data[feature_columns].values
    print(f"Features shape: {features.shape}, Labels shape: {labels.shape}")
    # Enforce expected feature count (32)
    if features.shape[1] != 32:
        print(f"Warning: Expected 32 features, got {features.shape[1]}. Check dataset columns and order.")
    unique_labels, counts = np.unique(labels, return_counts=True)
    n_classes = len(unique_labels)
    print("\n=== Class Label Distribution ===")
    for label, count in zip(unique_labels, counts):
        print(f"Label {int(label)}: {count} samples")
    # Class weights (informational)
    class_weights = {}
    n_samples = len(labels)
    for label, count in zip(unique_labels, counts):
        weight = n_samples / (n_classes * count)
        class_weights[int(label)] = weight
    print(f"Class weights: {class_weights}")
    # Signal-domain preprocessing
    print(f"Initial features shape: {features.shape}")
    features = np.apply_along_axis(wavelet_denoise, 0, features)
    features = bandpass_filter(features)
    features = np.clip(features, np.percentile(features, 2), np.percentile(features, 98))
    # Rolling smooth and moving averages
    feature_df = pd.DataFrame(features, columns=feature_columns)
    feature_df = feature_df.rolling(window=3).mean().bfill()
    print(f"Feature dataframe shape after rolling: {feature_df.shape}")
    data_ma = feature_df.rolling(window=5).mean().bfill()
    data_ma.columns = [f"{col}_ma" for col in feature_df.columns]
    enhanced_data = pd.concat([feature_df, data_ma], axis=1)
    print(f"Enhanced data shape: {enhanced_data.shape}, num_variates: {enhanced_data.shape[1]}")
    # Normalize enhanced features
    scaler_yeo = PowerTransformer(method="yeo-johnson", standardize=True)
    norm_data = scaler_yeo.fit_transform(enhanced_data)
    norm_data = np.nan_to_num(norm_data, nan=0.0, posinf=5.0, neginf=-5.0)
    norm_data = np.clip(norm_data, -10.0, 10.0)
    norm_data = pd.DataFrame(norm_data, columns=enhanced_data.columns)
    print(f"Normalized data shape: {norm_data.shape}, labels shape: {labels.shape}")
    # Quantum targets from mean of original features (pre-rolling)
    quantum_targets = np.mean(features, axis=1)
    quantum_scaler_global = PowerTransformer(method="yeo-johnson", standardize=True)
    quantum_targets = quantum_scaler_global.fit_transform(quantum_targets.reshape(-1, 1)).flatten()
    quantum_targets = np.nan_to_num(quantum_targets, nan=0.0, posinf=5.0, neginf=-5.0)
    quantum_targets = np.clip(quantum_targets, -10.0, 10.0)
    qmin, qmax = quantum_targets.min(), quantum_targets.max()
    qrange = qmax - qmin if qmax != qmin else 1
    quantum_targets = 2 * (quantum_targets - qmin) / qrange - 1
    print(f"Quantum targets range: [{quantum_targets.min():.4f}, {quantum_targets.max():.4f}]")
    print(f"Number of Target Values: {len(quantum_targets)}")
    print(f"Target Value Name: Quantum Target (Mean of Original Features)")
    # Align shapes if mismatch
    if norm_data.shape[0] != labels.shape[0] or enhanced_data.shape[0] != labels.shape[0] or len(quantum_targets) != labels.shape[0]:
        print(f"Shape mismatch detected: enhanced_data {enhanced_data.shape[0]}, norm_data {norm_data.shape[0]}, labels {labels.shape[0]}, quantum_targets {len(quantum_targets)}. Slicing to minimum length.")
        min_len = min(enhanced_data.shape[0], norm_data.shape[0], labels.shape[0], len(quantum_targets))
        enhanced_data = enhanced_data.iloc[:min_len]
        norm_data = norm_data.iloc[:min_len]
        labels = labels[:min_len]
        quantum_targets = quantum_targets[:min_len]
    # Config
    config = {
        "num_variates": norm_data.shape[1],
        "seq_len": 256,
        "pred_len": 1,
        "hidden_size": 512,
        "num_heads": 8,
        "num_layers": 4,
        "num_experts": 8,
        "dropout": 0.3,
        "lr": 2e-4,
        "batch_size": 64,
        "epochs": 80,
        "patience": 2,
        "grad_clip": 1.0,
        "warmup_epochs": 10,
        "grad_accum_steps": 1,
        "n_splits": 2,
        "min_samples": 299,
        "num_workers": 4,
        "optimizer": "adamw",
        "weight_decay": 5e-3,
        "scheduler_type": "cosine",
        "activation": "gelu",
        "ffn_multiplier": 4,
        "quantum_weight": 0.4,
        "class_weight": 0.6,
    }
    print(f"\nModel will use {config['num_variates']} input features and {n_classes} output classes")
    enhanced_data_np = enhanced_data.values
    kf = KFold(n_splits=config["n_splits"], shuffle=True)
    fold_results = []
    for fold, (train_idx, val_idx) in enumerate(kf.split(enhanced_data_np)):
        print(f"\n=== Fold {fold+1}/{config['n_splits']} ===")
        # Split
        train_data = enhanced_data.iloc[train_idx]
        val_data = enhanced_data.iloc[val_idx]
        train_labels = labels[train_idx]
        val_labels = labels[val_idx]
        train_quantum_targets = quantum_targets[train_idx]
        val_quantum_targets = quantum_targets[val_idx]
        print(f"Train data shape: {train_data.shape}, Train labels shape: {train_labels.shape}")
        print(f"Val data shape: {val_data.shape}, Val labels shape: {val_labels.shape}")
        # Fold-wise feature normalization
        scaler_yeo_train = PowerTransformer(method="yeo-johnson", standardize=True)
        norm_train = scaler_yeo_train.fit_transform(train_data)
        norm_train = np.nan_to_num(norm_train, nan=0.0, posinf=5.0, neginf=-5.0)
        norm_train = np.clip(norm_train, -10.0, 10.0)
        norm_train = pd.DataFrame(norm_train, columns=enhanced_data.columns)
        scaler_yeo_val = PowerTransformer(method="yeo-johnson", standardize=True)
        norm_val = scaler_yeo_val.fit_transform(val_data)
        norm_val = np.nan_to_num(norm_val, nan=0.0, posinf=5.0, neginf=-5.0)
        norm_val = np.clip(norm_val, -10.0, 10.0)
        norm_val = pd.DataFrame(norm_val, columns=enhanced_data.columns)
        # Fold-wise quantum target normalization to [-1,1]
        quantum_scaler = PowerTransformer(method="yeo-johnson", standardize=True)
        train_q_norm = quantum_scaler.fit_transform(train_quantum_targets.reshape(-1, 1)).flatten()
        train_q_norm = np.nan_to_num(train_q_norm, nan=0.0, posinf=5.0, neginf=-5.0)
        train_q_norm = np.clip(train_q_norm, -10.0, 10.0)
        tq_min, tq_max = train_q_norm.min(), train_q_norm.max()
        tq_range = tq_max - tq_min if tq_max != tq_min else 1
        train_q_norm = 2 * (train_q_norm - tq_min) / tq_range - 1
        val_q_norm = quantum_scaler.transform(val_quantum_targets.reshape(-1, 1)).flatten()
        val_q_norm = np.nan_to_num(val_q_norm, nan=0.0, posinf=5.0, neginf=-5.0)
        val_q_norm = np.clip(val_q_norm, -10.0, 10.0)
        val_q_norm = 2 * (val_q_norm - tq_min) / tq_range - 1
        # Datasets
        try:
            train_dataset = TimeSeriesDataset(norm_train, config["seq_len"], config["pred_len"], train_labels)
            val_dataset = TimeSeriesDataset(norm_val, config["seq_len"], config["pred_len"], val_labels)
            print(f"Fold {fold+1} - Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
        except Exception as e:
            print(f"Error creating datasets for fold {fold+1}: {e}. Skipping fold.")
            continue
        # DataLoaders
        train_loader = DataLoader(
            train_dataset,
            batch_size=config["batch_size"],
            shuffle=True,
            pin_memory=True,
            num_workers=config["num_workers"],
            persistent_workers=True if config["num_workers"] > 0 else False,
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=config["batch_size"],
            shuffle=False,
            pin_memory=True,
            num_workers=config["num_workers"],
            persistent_workers=True if config["num_workers"] > 0 else False,
        )
        if len(train_loader) == 0:
            print(f"Train loader empty for fold {fold+1}. Skipping fold.")
            continue
        # Model
        try:
            model = iTransformer(
                num_variates=config["num_variates"],
                seq_len=config["seq_len"],
                pred_len=config["pred_len"],
                n_classes=n_classes,
                hidden_size=config["hidden_size"],
                num_heads=config["num_heads"],
                num_layers=config["num_layers"],
                num_experts=config["num_experts"],
                dropout=config["dropout"],
                ffn_multiplier=config["ffn_multiplier"],
                activation=config["activation"],
                device=device,
            ).to(device)
            print(f"Model initialized on {device}")
        except Exception as e:
            print(f"Error initializing model for fold {fold+1}: {e}. Skipping fold.")
            continue
        optimizer = get_optimizer(model, config["optimizer"], config["lr"], config["weight_decay"])
        scheduler = get_scheduler(optimizer, config["scheduler_type"], config["warmup_epochs"], config["epochs"])
        criterion = ECG_loss(quantum_weight=config["quantum_weight"], class_weight=config["class_weight"]).to(device)
        grad_scaler = GradScaler("cuda")
        best_val_loss = float("inf")
        best_epoch = 0
        patience_counter = 0
        # Training
        for epoch in range(config["epochs"]):
            model.train()
            train_loss = 0.0
            for batch_idx, (x, y_quantum, y_label) in enumerate(tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}")):
                x = x.to(device, non_blocking=True)
                y_quantum = y_quantum.to(device, non_blocking=True)
                y_label = y_label.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    pred_quantum, pred_logits = model(x)
                    loss = criterion(pred_quantum, y_quantum, pred_logits, y_label)
                grad_scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_([p for n, p in model.named_parameters() if "weights" not in n], config["grad_clip"])
                torch.nn.utils.clip_grad_norm_([p for n, p in model.named_parameters() if "weights" in n], 0.3)
                grad_scaler.step(optimizer)
                grad_scaler.update()
                train_loss += loss.item()
                del x, y_quantum, y_label, pred_quantum, pred_logits, loss
            train_loss /= max(1, len(train_loader))
            # Validation
            model.eval()
            val_loss = 0.0
            pred_quantums, true_quantums = [], []
            pred_logits_list, true_labels_list = [], []
            with torch.no_grad():
                for x, y_quantum, y_label in val_loader:
                    x = x.to(device, non_blocking=True)
                    y_quantum = y_quantum.to(device, non_blocking=True)
                    y_label = y_label.to(device, non_blocking=True)
                    pred_quantum, pred_logits = model(x)
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        loss = criterion(pred_quantum, y_quantum, pred_logits, y_label)
                    val_loss += loss.item()
                    pred_quantums.append(pred_quantum.cpu())
                    true_quantums.append(y_quantum.cpu())
                    pred_logits_list.append(pred_logits.cpu())
                    true_labels_list.append(y_label.cpu())
                    del x, y_quantum, y_label, pred_quantum, pred_logits, loss
            val_loss /= max(1, len(val_loader))
            scheduler.step()
            pred_quantums = torch.cat(pred_quantums) if pred_quantums else torch.empty(0)
            true_quantums = torch.cat(true_quantums) if true_quantums else torch.empty(0)
            pred_logits = torch.cat(pred_logits_list) if pred_logits_list else torch.empty(0)
            true_labels = torch.cat(true_labels_list) if true_labels_list else torch.empty(0)
            # Compute metrics (using fold scaler to produce consistent display scale)
            if pred_quantums.numel():
                pred_q_denorm = quantum_scaler.inverse_transform(pred_quantums.numpy().reshape(-1, 1)).flatten()
                pred_q_denorm = 2 * (pred_q_denorm - tq_min) / tq_range - 1
                true_q_denorm = quantum_scaler.inverse_transform(true_quantums.numpy().reshape(-1, 1)).flatten()
                true_q_denorm = 2 * (true_q_denorm - tq_min) / tq_range - 1
                quantum_metrics = calculate_metrics(torch.tensor(true_q_denorm), torch.tensor(pred_q_denorm))
                quantum_rmse = quantum_metrics["Quantum_RMSE"]
                quantum_mae = quantum_metrics["Quantum_MAE"]
                quantum_conf_interval = quantum_metrics["Quantum_Conf_Interval"]
                quantum_corr = quantum_metrics["Quantum_Correlation"]
            else:
                quantum_rmse = quantum_mae = quantum_conf_interval = quantum_corr = float("nan")
            try:
                pred_probs = F.softmax(pred_logits, dim=1)
                roc_auc = roc_auc_score(true_labels.numpy(), pred_probs.numpy(), multi_class="ovr") if n_classes > 0 else float("nan")
            except Exception:
                roc_auc = float("nan")
            print(
                f"Fold {fold+1} Epoch {epoch+1}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}, "
                f"Quantum RMSE={quantum_rmse:.6f}, Quantum MAE={quantum_mae:.6f}, "
                f"Quantum Conf Interval={quantum_conf_interval:.6f}, Quantum Correlation={quantum_corr:.4f}, ROC AUC={roc_auc:.4f}"
            )
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                torch.save(model.state_dict(), f"best_model_fold{fold+1}.pth")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= config["patience"]:
                    print(f"Early stopping at fold {fold+1} epoch {epoch+1}")
                    break
        # Test best model on validation as proxy test
        print(f"\nLoading best model from fold {fold+1} epoch {best_epoch} for testing...")
        state_dict = torch.load(f"best_model_fold{fold+1}.pth", weights_only=True)
        model.load_state_dict(state_dict)
        model.eval()
        val_loss = 0.0
        pred_quantums, true_quantums, pred_logits_list, true_labels_list = [], [], [], []
        with torch.no_grad():
            for batch_idx, (x, y_quantum, y_label) in enumerate(val_loader):
                x = x.to(device, non_blocking=True)
                y_quantum = y_quantum.to(device, non_blocking=True)
                y_label = y_label.to(device, non_blocking=True)
                pred_quantum, pred_logits = model(x)
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    loss = criterion(pred_quantum, y_quantum, pred_logits, y_label)
                val_loss += loss.item()
                pred_quantums.append(pred_quantum.cpu())
                true_quantums.append(y_quantum.cpu())
                pred_logits_list.append(pred_logits.cpu())
                true_labels_list.append(y_label.cpu())
                del x, y_quantum, y_label, pred_quantum, pred_logits, loss
        val_loss /= max(1, len(val_loader))
        pred_quantums = torch.cat(pred_quantums)
        true_quantums = torch.cat(true_quantums)
        pred_logits = torch.cat(pred_logits_list)
        true_labels = torch.cat(true_labels_list)
        # Denorm for metrics
        pred_q_denorm = quantum_scaler.inverse_transform(pred_quantums.numpy().reshape(-1, 1)).flatten()
        pred_q_denorm = 2 * (pred_q_denorm - tq_min) / tq_range - 1
        true_q_denorm = quantum_scaler.inverse_transform(true_quantums.numpy().reshape(-1, 1)).flatten()
        true_q_denorm = 2 * (true_q_denorm - tq_min) / tq_range - 1
        test_metrics = calculate_metrics(torch.tensor(true_q_denorm), torch.tensor(pred_q_denorm))
        quantum_test_rmse = test_metrics["Quantum_RMSE"]
        quantum_test_mae = test_metrics["Quantum_MAE"]
        quantum_conf_interval = test_metrics["Quantum_Conf_Interval"]
        quantum_corr = test_metrics["Quantum_Correlation"]
        try:
            pred_probs = F.softmax(pred_logits, dim=1)
            roc_auc = roc_auc_score(true_labels.numpy(), pred_probs.numpy(), multi_class="ovr") if n_classes > 0 else float("nan")
        except Exception:
            roc_auc = float("nan")
        print(f"\n=== FOLD {fold+1} TEST RESULTS ===")
        print(f"Test Loss: {val_loss:.6f}")
        print(f"Quantum Test RMSE: {quantum_test_rmse:.6f}")
        print(f"Quantum Test MAE: {quantum_test_mae:.6f}")
        print(f"Quantum Confidence Interval: {quantum_conf_interval:.6f}")
        print(f"Quantum Correlation Coefficient: {quantum_corr:.4f}")
        print(f"ROC AUC: {roc_auc:.4f}")
        if val_loss < 0.5:
            print("Test loss goal achieved!")
        else:
            print("Test loss still high, consider further tuning.")
        fold_results.append({
            "fold": fold + 1,
            "test_loss": val_loss,
            "quantum_rmse": quantum_test_rmse,
            "quantum_mae": quantum_test_mae,
            "quantum_conf_interval": quantum_conf_interval,
            "quantum_correlation": quantum_corr,
            "roc_auc": roc_auc,
        })
        # Cleanup
        del train_dataset, val_dataset, train_loader, val_loader, model
        torch.cuda.empty_cache()
        gc.collect()
    # Cross-validation summary
    if fold_results:
        print("\n=== CROSS-VALIDATION SUMMARY ===")
        avg_test_loss = np.mean([r["test_loss"] for r in fold_results])
        avg_quantum_correlation = np.mean([r["quantum_correlation"] for r in fold_results])
        avg_roc_auc = np.nanmean([r["roc_auc"] for r in fold_results])
        print(f"Average Test Loss: {avg_test_loss:.6f}")
        print(f"Average Quantum Correlation: {avg_quantum_correlation:.4f}")
        print(f"Average ROC AUC: {avg_roc_auc:.4f}")
        for r in fold_results:
            print(
                f"Fold {r['fold']}: Test Loss={r['test_loss']:.6f}, "
                f"Quantum Correlation={r['quantum_correlation']:.4f}, ROC AUC={r['roc_auc']:.4f}"
            )
        best_fold = min(fold_results, key=lambda x: x["test_loss"])
        print(f"\n=== BEST FOLD SELECTED ===")
        print(f"Best Fold: {best_fold['fold']}")
        print(f"Test Loss: {best_fold['test_loss']:.6f}")
        print(f"Quantum Test RMSE: {best_fold['quantum_rmse']:.6f}")
        print(f"Quantum Test MAE: {best_fold['quantum_mae']:.6f}")
        print(f"Quantum Confidence Interval: {best_fold['quantum_conf_interval']:.6f}")
        print(f"Quantum Correlation Coefficient: {best_fold['quantum_correlation']:.4f}")
        print(f"ROC AUC: {best_fold['roc_auc']:.4f}")
        print(f"Best model weights saved at: best_model_fold{best_fold['fold']}.pth")
    else:
        print("No valid folds processed. Check dataset size or split configuration.")

if __name__ == "__main__":
    main()


PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
Device: cuda:0
Raw data shape after dropping NaN: (149503, 33)
Features shape: (149503, 32), Labels shape: (149503,)

=== Class Label Distribution ===
Label 0: 57703 samples
Label 1: 30834 samples
Label 2: 31756 samples
Label 3: 29210 samples
Class weights: {0: 0.6477262880612793, 1: 1.212160277615619, 2: 1.1769665575009447, 3: 1.279553235193427}
Initial features shape: (149503, 32)
Feature dataframe shape after rolling: (149504, 32)
Enhanced data shape: (149504, 64), num_variates: 64
Normalized data shape: (149504, 64), labels shape: (149503,)
Quantum targets range: [-1.0000, 1.0000]
Number of Target Values: 149504
Target Value Name: Quantum Target (Mean of Original Features)
Shape mismatch detected: enhanced_data 149504, norm_data 149504, labels 149503, quantum_targets 149504. Slicing to minimum length.

Model will use 64 input features and 4 output classes

=== Fold 1/2 ===
Train data shape: (74751, 64), Train labe

Fold 1 Epoch 1: 100%|██████████| 1164/1164 [04:58<00:00,  3.90it/s]


Fold 1 Epoch 1: Train Loss=0.432744, Val Loss=0.250388, Quantum RMSE=0.013597, Quantum MAE=0.010525, Quantum Conf Interval=0.000089, Quantum Correlation=0.8598, ROC AUC=0.9658


Fold 1 Epoch 2: 100%|██████████| 1164/1164 [04:57<00:00,  3.91it/s]


Fold 1 Epoch 2: Train Loss=0.238649, Val Loss=0.179710, Quantum RMSE=0.014662, Quantum MAE=0.012112, Quantum Conf Interval=0.000078, Quantum Correlation=0.8760, ROC AUC=0.9855


...

Fold 2 Epoch 6: Train Loss=0.039268, Val Loss=0.044930, Quantum RMSE=0.007107, Quantum MAE=0.005428, Quantum Conf Interval=0.000042, Quantum Correlation=0.9611, ROC AUC=0.9979


Fold 2 Epoch 7: 100%|██████████| 1164/1164 [04:58<00:00,  3.90it/s]


Fold 2 Epoch 7: Train Loss=0.034261, Val Loss=0.046514, Quantum RMSE=0.007093, Quantum MAE=0.005365, Quantum Conf Interval=0.000043, Quantum Correlation=0.9604, ROC AUC=0.9979


Fold 2 Epoch 8: 100%|██████████| 1164/1164 [04:57<00:00,  3.91it/s]


Fold 2 Epoch 8: Train Loss=0.034236, Val Loss=0.045255, Quantum RMSE=0.006982, Quantum MAE=0.005200, Quantum Conf Interval=0.000044, Quantum Correlation=0.9612, ROC AUC=0.9981
Early stopping at fold 2 epoch 8

Loading best model from fold 2 epoch 6 for testing...

=== FOLD 2 TEST RESULTS ===
Test Loss: 0.044930
Quantum Test RMSE: 0.007107
Quantum Test MAE: 0.005428
Quantum Confidence Interval: 0.000042
Quantum Correlation Coefficient: 0.9611
ROC AUC: 0.9979
Test loss goal achieved!

=== CROSS-VALIDATION SUMMARY ===
Average Test Loss: 0.044252
Average Quantum Correlation: 0.9637
Average ROC AUC: 0.9980
Fold 1: Test Loss=0.043573, Quantum Correlation=0.9662, ROC AUC=0.9981
Fold 2: Test Loss=0.044930, Quantum Correlation=0.9611, ROC AUC=0.9979

=== BEST FOLD SELECTED ===
Best Fold: 1
Test Loss: 0.043573
Quantum Test RMSE: 0.005466
Quantum Test MAE: 0.003900
Quantum Confidence Interval: 0.000039
Quantum Correlation Coefficient: 0.9662
ROC AUC: 0.9981
Best model weights saved at: best_model

# Dataset information

In [9]:
import pandas as pd
import numpy as np
import io

# Your requested dataset path
DATASET_PATH = "/kaggle/input/two-lead-test/DATASET_CSV.csv"

def inspect_dataset(csv_path: str):
    print(f"Loading dataset from: {csv_path}")
    df = pd.read_csv(csv_path)
    
    # Basic shape
    print("\n=== Basic shape ===")
    print(f"shape: {df.shape}")  # (rows, columns)
    
    # Columns and dtypes
    print("\n=== Columns (all) ===")
    print(list(df.columns))
    
    print("\n=== dtypes ===")
    print(df.dtypes)
    
    # Info summary
    print("\n=== .info() ===")
    buf = io.StringIO()
    df.info(buf=buf)
    print(buf.getvalue())  # concise summary [16]
    
    # Preview rows
    print("\n=== .head(5) ===")
    print(df.head(5))  # quick preview [9]
    
    # Target presence and distribution
    has_type = 'type' in df.columns
    print(f"\nTarget column present (type): {has_type}")
    if has_type:
        print("\n=== Label distribution (type) ===")
        print(df['type'].value_counts(dropna=False))  # label counts [1]
    
    # Proposed feature columns (exclude target)
    feature_cols = [c for c in df.columns if c != 'type']
    print("\n=== Proposed feature columns (excluding 'type') ===")
    print(feature_cols)
    print(f"Proposed feature count: {len(feature_cols)}")
    
    # Check expected 32 features (excluding type)
    if len(feature_cols) != 32:
        print(f"WARNING: Expected 32 feature columns excluding 'type', but found {len(feature_cols)}.")
        # Help diagnose by listing first N columns with indices
        print("\nFirst 40 columns with indices to help selection:")
        for i, c in enumerate(df.columns[:40]):
            print(f"{i:>3}: {c}")
    
    # Numeric vs non-numeric in features
    num_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    nonnum_cols = [c for c in feature_cols if c not in num_cols]
    print("\n=== Feature type summary ===")
    print(f"Numeric feature columns ({len(num_cols)}): {num_cols[:20]}{' ...' if len(num_cols) > 20 else ''}")
    print(f"Non-numeric feature columns ({len(nonnum_cols)}): {nonnum_cols}")
    
    # Descriptive statistics for numeric features
    if num_cols:
        print("\n=== Descriptive statistics (numeric features) ===")
        print(df[num_cols].describe().T)  # statistical summary [14]
    
    # Final summary for model wiring (fixed indices)
    print("\n=== Summary for model wiring ===")
    print(f"- Total rows: {df.shape[0]}")
    print(f"- Total columns: {df.shape[1]}")
    print(f"- Target column: {'type' if has_type else 'MISSING'}")
    print(f"- Feature columns (excluding 'type'): {len(feature_cols)} (expected 32)")
    if len(feature_cols) == 32 and has_type:
        print("OK: Dataset fits the expected format: 32 features + 1 target 'type'.")

if __name__ == "__main__":
    inspect_dataset(DATASET_PATH)

Loading dataset from: /kaggle/input/two-lead-test/DATASET_CSV.csv

=== Basic shape ===
shape: (149505, 33)

=== Columns (all) ===
['type', '0_pre-RR', '0_post-RR', '0_pPeak', '0_tPeak', '0_rPeak', '0_sPeak', '0_qPeak', '0_qrs_interval', '0_pq_interval', '0_qt_interval', '0_st_interval', '0_qrs_morph0', '0_qrs_morph1', '0_qrs_morph2', '0_qrs_morph3', '0_qrs_morph4', '1_pre-RR', '1_post-RR', '1_pPeak', '1_tPeak', '1_rPeak', '1_sPeak', '1_qPeak', '1_qrs_interval', '1_pq_interval', '1_qt_interval', '1_st_interval', '1_qrs_morph0', '1_qrs_morph1', '1_qrs_morph2', '1_qrs_morph3', '1_qrs_morph4']

=== dtypes ===
type                int64
0_pre-RR            int64
0_post-RR           int64
0_pPeak           float64
0_tPeak           float64
0_rPeak           float64
0_sPeak           float64
0_qPeak           float64
0_qrs_interval      int64
0_pq_interval       int64
0_qt_interval       int64
0_st_interval       int64
0_qrs_morph0      float64
0_qrs_morph1      float64
0_qrs_morph2      float

# Package

In [ ]:
!pip uninstall jax jaxlib -y
!pip install jax==0.4.28  jaxlib==0.4.28
!pip install --upgrade scikit-learn imbalanced-learn
!pip install pennylane==0.40.0 pennylane-lightning[gpu]==0.40.0
!pip install autoray==0.6.11
!pip install captum
